# chess-vi — SFT trên Colab

Notebook này **chỉ gọi script** `chessvi.train.sft`. Không copy-paste logic vào cell:
logic nằm trong repo để test được và để Colab với local chạy đúng một thứ.

Runtime cần: **A100 / L4 / T4 GPU**. Colab hay ngắt giữa chừng nên script bật
`hub_strategy="checkpoint"` — cell cuối resume lại từ checkpoint.

Thứ tự chạy: `Mount` → `Cài đặt` → `Token` → `GPU` → `Validate` → `Smoke test` → `Train`.
Đứt kết nối thì chạy lại 4 cell đầu rồi nhảy thẳng xuống cell **Resume**.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/chessvi"
!mkdir -p {DRIVE_ROOT}/outputs

## 2. Lấy repo và cài đặt

Sửa `REPO_URL` thành remote của bạn. Nếu đã clone repo vào Drive thì chỉ cần `cd`.

In [ ]:
REPO_URL = "https://github.com/trantrien1/ChessVi.git"
REPO_DIR = "/content/ChessVi"

import os

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull --ff-only || true

!pip install -q -e ".[train,data]"
!pip install -q bitsandbytes==0.50.2

## 3. Token Hugging Face

Lưu token trong **Colab Secrets** (biểu tượng chìa khoá, tên `HF_TOKEN`).
Không bao giờ dán token thẳng vào cell — notebook sẽ bị commit kèm token.

In [ ]:
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.environ["HF_TOKEN"], "chưa đặt secret HF_TOKEN"

## 4. Kiểm tra GPU

In [ ]:
!nvidia-smi

import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

## 5. Validate dữ liệu đã dịch (T6)

`--out-clean` ghi tập đã pass ra `data/validated/sft` — đây chính là đầu vào
của SFT. Không có bước này thì không có file nào chứa "dữ liệu đã validate".

Kỳ vọng loại 5–15%. Trên 25% script sẽ in cảnh báo to: pipeline dịch có vấn đề,
quay lại T5 chứ đừng train.

**Chốt chặn thủ công:** trước khi sang cell tiếp theo, tự đọc tay 200 mẫu ngẫu
nhiên đã pass. Không ai thay bạn làm bước này được.

In [ ]:
TRANSLATED = f"{DRIVE_ROOT}/data/translated/sft"
DATA = f"{DRIVE_ROOT}/data/validated/sft"

!python -m chessvi.data.validate \
    --input {TRANSLATED} \
    --rejected {DRIVE_ROOT}/data/rejected.jsonl \
    --out-clean {DATA}

!ls -la {DATA}

## 6. Smoke test

5 step với model 0.6B. Cell này phải chạy xong không lỗi **trước khi** tốn CU
cho lần train thật.

In [ ]:
!python -m chessvi.train.sft \
    --data {DATA} \
    --base-model Qwen/Qwen3-0.6B \
    --limit 50 --max-steps 5 \
    --output-dir /content/outputs/smoke \
    --no-push

## 7. Train thật

Đổi `HUB_MODEL_ID` thành repo của bạn. Repo được tạo ở chế độ **private**.

In [ ]:
HUB_MODEL_ID = "trantrien1/chessvi-4b-sft"
OUTPUT_DIR = f"{DRIVE_ROOT}/outputs/sft"

!python -m chessvi.train.sft \
    --data {DATA} \
    --base-model Qwen/Qwen3-4B \
    --output-dir {OUTPUT_DIR} \
    --hub-model-id {HUB_MODEL_ID} \
    --epochs 2 --batch-size 1 --grad-accum 16 --save-steps 200

## 8. Resume sau khi Colab ngắt

Chạy lại cell 1–4 rồi chạy cell này. `--resume` đọc checkpoint mới nhất trong
`--output-dir`; nếu output nằm trên Drive thì checkpoint vẫn còn nguyên.

In [ ]:
!ls -la {OUTPUT_DIR} | head -20

!python -m chessvi.train.sft \
    --data {DATA} \
    --base-model Qwen/Qwen3-4B \
    --output-dir {OUTPUT_DIR} \
    --hub-model-id {HUB_MODEL_ID} \
    --epochs 2 --batch-size 1 --grad-accum 16 --save-steps 200 \
    --resume

## 9. Chốt chặn sau T8

Accuracy trên test set phải đạt **30–38%**. Dưới 20% nghĩa là dữ liệu có vấn đề
— quay lại T5, **đừng** chạy RL.

```bash
python -m chessvi.eval.puzzle_acc --puzzles data/puzzles/test.parquet \
    --backend hf --model-path outputs/sft --out reports/sft_acc.csv
```